In [1]:
import pandas as pd
import numpy as np
import json
import os

# 配置
data_dir = "../eval/eval_results"  # 替换为你的实际路径
base_name = "math_instruct_128_64_{}_generations.json"
num_files = 8  # 从 0 到 7

all_generations = []

for i in range(num_files):
    filename = base_name.format(i)
    file_path = os.path.join(data_dir, filename)
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        # 假设每个文件是 {'generations': [...]}
        if 'generations' not in data:
            raise KeyError(f"'generations' key not found in {filename}")
        all_generations.extend(data['generations'])

# 最终合并结果：all_generations 是一个大列表，包含所有样本
print(f"✅ Total samples loaded: {len(all_generations)}")

# 如果你希望保持和之前一样的结构（即 results = {'generations': [...] }）
results = {'generations': all_generations}


✅ Total samples loaded: 500


In [20]:
from typing import Optional, List, Dict

# ---------- 官方答案提取函数 ----------
def last_boxed_only_string(string: str) -> Optional[str]:
    """Extract the last \\boxed{...} content, handling nested braces."""
    idx = string.rfind("\\boxed")
    if idx < 0:
        return None

    i = idx + 7  # len("\\boxed{") == 7
    if i >= len(string):
        return None

    nest_level = 0
    for j in range(i, len(string)):
        if string[j] == "{":
            nest_level += 1
        elif string[j] == "}":
            if nest_level == 0:
                return string[i:j]
            nest_level -= 1
    return None


# ---------- 官方答案标准化 ----------
def normalize_final_answer(final_answer: str) -> str:
    """Normalize answer string for non-symbolic comparison (e.g., sets)."""
    final_answer = final_answer.strip()
    
    # Remove \text{...}
    final_answer = re.sub(r'\\text\{([^}]*)\}', r'\1', final_answer)
    
    # Normalize fractions
    final_answer = re.sub(r'\\frac\s*\{\s*(.*?)\s*\}\s*\{\s*(.*?)\s*\}', r'\\frac{\1}{\2}', final_answer)
    
    # Collapse whitespace
    final_answer = re.sub(r'\s+', ' ', final_answer)
    
    # Sort simple numeric sets
    if final_answer.startswith(r'\{') and final_answer.endswith(r'\}'):
        try:
            inner = final_answer[2:-2].strip()
            if inner:
                elements = [x.strip() for x in inner.split(',')]
                nums = []
                for e in elements:
                    try:
                        nums.append(float(e))
                    except ValueError:
                        nums = None
                        break
                if nums is not None:
                    nums.sort()
                    sorted_inner = ', '.join(
                        str(int(x)) if x.is_integer() else str(x) for x in nums
                    )
                    final_answer = r'\{' + sorted_inner + r'\}'
        except Exception:
            pass
    
    return final_answer


# ---------- 官方等价判断 ----------
def is_equiv(str1: Optional[str], str2: Optional[str], verbose: bool = False) -> bool:
    if str1 is None and str2 is None:
        return True
    if str1 is None or str2 is None:
        return False

    # Try LaTeX parsing first
    try:
        from latex2sympy2 import latex2sympy
        expr1 = latex2sympy(str1)
        expr2 = latex2sympy(str2)
        
        # Exact symbolic equality
        if expr1 == expr2:
            return True
        
        # Use SymPy's equals for more robust comparison
        if hasattr(expr1, 'equals') and expr1.equals(expr2):
            return True
        
        # Numerical fallback
        try:
            if abs(float(expr1.evalf()) - float(expr2.evalf())) < 1e-6:
                return True
        except Exception:
            pass
            
    except Exception as e1:
        if verbose:
            print(f"LaTeX parsing failed: {e1}")
        try:
            # Fallback to sympify (for simple expressions)
            expr1 = sympify(str1.replace("\\", ""), evaluate=True)
            expr2 = sympify(str2.replace("\\", ""), evaluate=True)
            if expr1 == expr2:
                return True
            if hasattr(expr1, 'equals') and expr1.equals(expr2):
                return True
            if abs(float(expr1.evalf()) - float(expr2.evalf())) < 1e-6:
                return True
        except Exception as e2:
            if verbose:
                print(f"Sympify fallback failed: {e2}")
            pass

    # Final fallback: normalized string comparison
    return normalize_final_answer(str1) == normalize_final_answer(str2)



In [21]:
def evaluate_math_results(
    results: Dict[str, List[Dict]]
) -> Dict:
    """
    Evaluate MATH predictions using official logic.
    
    Args:
        results: Dictionary with key 'generations', value is list of samples
                 Each sample has 'generations' (model output) and 'ground_truth'
    
    Returns:
        dict with accuracy, correct count, total count, and per-sample details
    """
    generations_list = results['generations']
    
    predictions = []
    ground_truths = []
    correct = 0
    total = len(generations_list)
    
    for sample in generations_list:
        gen_text = sample['generations']
        gt = sample['ground_truth']
        
        pred = last_boxed_only_string(gen_text)
        predictions.append(pred)
        ground_truths.append(gt)
        
        equiv = is_equiv(pred, gt)
        if equiv:
            correct += 1
    
    accuracy = correct / total if total > 0 else 0.0
    
    return {
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
        "predictions": predictions,
        "ground_truths": ground_truths,
        "details": [
            {
                "question": s["question"],
                "prediction": pred,
                "ground_truth": gt,
                "is_correct": equiv
            }
            for s, pred, gt, equiv in zip(generations_list, predictions, ground_truths, [is_equiv(p, g) for p, g in zip(predictions, ground_truths)])
        ]
    }

In [22]:
eval_result = evaluate_math_results(results)
    
print(f"✅ MATH Accuracy: {eval_result['accuracy']:.2%} ({eval_result['correct']}/{eval_result['total']})")

# 查看前几个结果
for item in eval_result['details'][:5]:
    print(f"\nQuestion: {item['question']}")
    print(f"Pred: {item['prediction']}")
    print(f"GT: {item['ground_truth']}")
    print(f"Correct: {item['is_correct']}")

✅ MATH Accuracy: 26.00% (130/500)

Question: Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$
Pred:  (3, \frac{\pi}{2})
GT: \left( 3, \frac{\pi}{2} \right)
Correct: False

Question: What is the distance, in units, between the points $(2, -6)$ and $(-4, 3)$? Express your answer in simplest radical form.
Pred: \sqrt{117}
GT: 3\sqrt{13}
Correct: True

Question: Compute: $1-2+3-4+5- \dots +99-100$.
Pred: None
GT: -50
Correct: False

Question: A worker receives an annual wage of $\$20{,}000$, which he always deposits into a savings account at the end of the year. By the end of the third year (when he makes the third deposit), he wants to have at least $\$66,200$ in the account to finance the purchase of a house. What is the minimal compound interest rate that the savings account must provide? Express your answer as a percentage, but do not include the percent sign.
Pred: 3.5
GT: 

In [10]:
!pip install sympy latex2sympy2

Looking in indexes: https://mirrors.ustc.edu.cn/pypi/web/simple
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.7.2-py3-none-any.whl size=140969 sha256=e0e9204faf15e12cfd16186dcf79bbdb0eaf05cba8effd5d586a5e3ef0238030
  Stored in directory: /root/.cache/pip/wheels/c0/4a/e1/e04f2beb560199148f0eeec63e164995033214e642cb76e3b0
Successfully built antlr4-python3-runtime
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [latex2sympy2]
